In [1]:
import kagglehub
import pandas as pd
import numpy as np
import os
import warnings
# from google.colab import drive

# Supresión de warnings
# warnings.filterwarnings('ignore')
# pd.options.mode.chained_assignment = None

# Configuración de visualización del DataFrame
pd.set_option('display.max_columns', None)   # muestra todas las columnas
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)


In [2]:
# Montar Google Drive (te va a pedir autorización la primera vez)
# drive.mount('/content/drive')


In [3]:

# ── Configuración ──────────────────────────────────────────
DATA_PATH_LOCAL         = r"data/raw/huracan_plantilla_raw.csv"                  # ← CAMBIAR: ruta local
DRIVE_PATH = "/MyDrive/trabajo/ds/huracan/huracan_plantilla_raw.csv"  # ← CAMBIAR: ruta dentro de tu Drive
# ───────────────────────────────────────────────────────────

df_raw = None

# Prioridad 1: archivo local ya descargado
if os.path.exists(DATA_PATH_LOCAL):
    df_raw = pd.read_csv(DATA_PATH_LOCAL)
    print(f"✅ Cargado desde archivo local: {DATA_PATH_LOCAL}")

# Prioridad 2: Google Drive
if df_raw is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        ruta_completa = f"/content/drive{DRIVE_PATH}"
        if os.path.exists(ruta_completa):
            df_raw = pd.read_csv(ruta_completa)
            print(f"✅ Cargado desde Google Drive: {ruta_completa}")
            # Guardar localmente para próximas ejecuciones
            os.makedirs(os.path.dirname(DATA_PATH_LOCAL), exist_ok=True)
            df_raw.to_csv(DATA_PATH_LOCAL, index=False)
            print(f"💾 Raw guardado localmente: {DATA_PATH_LOCAL}")
        else:
            print(f"⚠️  Archivo no encontrado en Drive: {ruta_completa}")
    except Exception as e:
        print(f"⚠️  No se pudo montar Drive: {e}")

✅ Cargado desde archivo local: data/raw/huracan_plantilla_raw.csv


In [4]:
# ─── Primeras y últimas filas ──────────────────────────────────────────────
print('\n--- Primeras 5 filas ---')
display(df_raw.head())

print('\n--- Últimas 5 filas ---')
display(df_raw.tail())

# ─── Dimensiones ───────────────────────────────────────────────────────────
print(f'Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}')

# ─── Tipos de datos ────────────────────────────────────────────────────────
print('\n--- Info general ---')
df_raw.info()

# ─── Valores nulos por columna ─────────────────────────────────────────────
print('\n--- Nulos por columna ---')
# Verificamos si hay algún nulo en toda la tabla
if df_raw.isnull().sum().sum() == 0:
    print('✅ Sin valores nulos')
else:
    # Sino mostramos tabla filtrada de nulos
    nulos = pd.DataFrame({
        'Nulos': df_raw.isnull().sum(),
        'Porcentaje': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
    })
    display(nulos[nulos['Nulos'] > 0])

# ─── Duplicados ────────────────────────────────────────────────────────────
print(f'\nFilas duplicadas: {df_raw.duplicated().sum()}')

# ─── Estadísticas descriptivas ─────────────────────────────────────────────
print('\n--- Estadísticas descriptivas ---')
display(df_raw.describe())


--- Primeras 5 filas ---


,#,Jugadores,F. Nacim./Edad,Nac.,Valor de mercado
0,32,Sebastián Meza Portero,14/03/2000 (26),NaN,450 mil €
1,NaN,Sebastián Meza,NaN,NaN,NaN
2,NaN,Portero,NaN,NaN,NaN
3,1,Hernán Galíndez Portero,30/03/1987 (39),NaN,300 mil €
4,NaN,Hernán Galíndez,NaN,NaN,NaN



--- Últimas 5 filas ---


,#,Jugadores,F. Nacim./Edad,Nac.,Valor de mercado
85,NaN,Eric Ramírez,NaN,NaN,NaN
86,NaN,Delantero centro,NaN,NaN,NaN
87,18,Luciano Giménez Delantero centro,18/02/2000 (26),NaN,300 mil €
88,NaN,Luciano Giménez,NaN,NaN,NaN
89,NaN,Delantero centro,NaN,NaN,NaN


Filas: 90 | Columnas: 5

--- Info general ---
<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   #                 30 non-null     str    
 1   Jugadores         90 non-null     str    
 2   F. Nacim./Edad    30 non-null     str    
 3   Nac.              0 non-null      float64
 4   Valor de mercado  30 non-null     str    
dtypes: float64(1), str(4)
memory usage: 3.6 KB

--- Nulos por columna ---


,Nulos,Porcentaje
#,60,66.67
F. Nacim./Edad,60,66.67
Nac.,90,100.00
Valor de mercado,60,66.67



Filas duplicadas: 18

--- Estadísticas descriptivas ---


,Nac.
count,0.00
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


In [5]:
# ── Eliminar filas con nulos ───────────────────────────────────────────────

df_raw = df_raw.dropna(subset=['#'])

In [6]:
# Reemplazar '-' por NaN real en todo el dataframe
df_raw = df_raw.replace('-', np.nan)

In [7]:
# Ver estado actual de nulos
nulos = pd.DataFrame({
    'Nulos': df_raw.isnull().sum(),
    'Porcentaje': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'Tipo': df_raw.dtypes
})
display(nulos[nulos['Nulos'] > 0].sort_values('Porcentaje', ascending=False))

,Nulos,Porcentaje,Tipo
Nac.,30,100.00,float64
#,2,6.67,str
Valor de mercado,1,3.33,str


# ───── Decisiones de imputación — Plantel Huracán ─────
- Valor de mercado: 1 NaN (Nazareno Durán) — se deja como NaN. Transfermarkt no tiene dato disponible ('-' en el sitio original)
pandas ignora NaN en .mean(), el promedio del plantel no se ve afectado
- Nac.: si bien tiene más de 50% nulos, la dejo para imputarle datos manualmente

In [8]:

# Caso A — Separar texto con patrón conocido al final del string (nombre + posición)

# ── Definir los valores posibles de la parte a extraer ───────────────────
# Listar de más largo a más corto para que el regex priorice el match completo
valores_a_extraer = [
    'Mediocentro ofensivo',     # ← reemplazar con los valores del dataset
    'Lateral izquierdo',
    'Lateral derecho',
    'Interior derecho',
    'Interior izquierdo',
    'Extremo izquierdo',
    'Extremo derecho',
    'Delantero centro',
    'Defensa central',
    'Mediocentro',
    'Portero',
    'Pivote',
]

patron = '(' + '|'.join(valores_a_extraer) + ')'

# ── Extraer en columna nueva ──────────────────────────────────────────────
df_raw['posicion'] = df_raw['Jugadores'].str.extract(patron)             # ← reemplazar nombre columna origen

# ── Limpiar la columna original dejando solo el nombre ───────────────────
df_raw['jugador'] = df_raw['Jugadores'].str.replace(patron, '', regex=True).str.strip()

# ── Verificar que no quedaron NaN en posicion ────────────────────────────
print(f"NaN en posicion: {df_raw['posicion'].isnull().sum()}")
display(df_raw[['jugador', 'posicion']].head(10))

# ── Eliminar columna original una vez verificado ─────────────────────────
df_raw = df_raw.drop(columns=['Jugadores'])                              # ← reemplazar

NaN en posicion: 0


,jugador,posicion
0,Sebastián Meza,Portero
3,Hernán Galíndez,Portero
6,Nazareno Durán,Portero
9,Lucas Carrizo,Defensa central
12,Hugo Nervo,Defensa central
15,Mauro Villar,Defensa central
18,Nehuén Paz,Defensa central
21,Fabio Pereyra,Defensa central
24,Máximo Palazzo,Defensa central
27,Daniel Zabala,Defensa central


In [9]:
# ── Separar en dos columnas usando regex con grupos ──────────────────────
# Ejemplo: '14/03/2000 (26)' → fecha_nacimiento='14/03/2000', edad=26
df_raw['fecha_nacimiento'] = df_raw['F. Nacim./Edad'].str.extract(r'(\d{2}/\d{2}/\d{4})')
df_raw['edad'] = df_raw['F. Nacim./Edad'].str.extract(r'\((\d+)\)').astype(float)

In [10]:
# ── Eliminar columna original una vez verificado ─────────────────────────
df_raw = df_raw.drop(columns=['F. Nacim./Edad'])

 #comento el bloque de código dado que ya se ejecutó y se borraron/agregaron columnas
 

In [11]:
# ── Convertir fecha a tipo datetime ──────────────────────────────────────
df_raw['fecha_nacimiento'] = pd.to_datetime(df_raw['fecha_nacimiento'],
                                             format='%d/%m/%Y',
                                             errors='coerce')
# ── Numéricos ────────────────────────────────────────────────────────────
df_raw['#'] = pd.to_numeric(df_raw['#'], errors='coerce').astype('Int64')
df_raw['edad'] = df_raw['edad'].astype(int)

# ── Numérico con abreviaciones de magnitud ────────────────────────────────
df_raw['Valor de mercado'] = (
    df_raw['Valor de mercado']
    .str.replace('€', '', regex=False)
    .str.replace('mill.', 'e6', regex=False)
    .str.replace('mil', 'e3', regex=False)
    .str.replace(',', '.', regex=False)
    .str.replace(' ', '', regex=False) 
    .str.strip()
)
df_raw['Valor de mercado'] = pd.to_numeric(df_raw['Valor de mercado'], errors='coerce')


In [12]:
# Verificar tipos después de convertir
print(df_raw.dtypes)

#                            Int64
Nac.                       float64
Valor de mercado           float64
posicion                       str
jugador                        str
fecha_nacimiento    datetime64[us]
edad                         int64
dtype: object


In [13]:
# Ver si el índice tiene huecos
print(f'Índice antes del reset: {df_raw.index.tolist()[:10]}...')

# Resetear
df_raw = df_raw.reset_index(drop=True)

# Verificar
print(f'Índice después del reset: {df_raw.index.tolist()[:10]}...')
print(f'Filas finales: {df_raw.shape[0]}')

Índice antes del reset: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27]...
Índice después del reset: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]...
Filas finales: 30


In [14]:
print('=' * 50)
print('REPORTE FINAL DE CALIDAD')
print('=' * 50)
print(f'Filas finales:      {df_raw.shape[0]}')
print(f'Columnas:           {df_raw.shape[1]}')
print(f'Duplicados:         {df_raw.duplicated().sum()}')
print(f'Nulos totales:      {df_raw.isnull().sum().sum()}')
print('=' * 50)

REPORTE FINAL DE CALIDAD
Filas finales:      30
Columnas:           7
Duplicados:         0
Nulos totales:      33


In [15]:
print(df_raw['Valor de mercado'].head(10))
print(df_raw['Valor de mercado'].dtype)

0   450000.00
1   300000.00
2         NaN
3   800000.00
4   225000.00
5   175000.00
6   175000.00
7   150000.00
8   125000.00
9   100000.00
Name: Valor de mercado, dtype: float64
float64


In [17]:
# Renombrar a df_clean antes de exportar
df_clean = df_raw.copy()

# Exportar
nombre_salida = 'data/clean/huracan_plantilla_clean.csv'    # ← reemplazar
df_clean.to_csv(nombre_salida, index=False, encoding='utf-8')
print(f'Dataset exportado: {nombre_salida}')
print(f'Filas: {df_clean.shape[0]} | Columnas: {df_clean.shape[1]}')

Dataset exportado: data/clean/huracan_plantilla_clean.csv
Filas: 30 | Columnas: 7
